# ✍️ 03 — Handwritten Text Recognition (HTR)

**EvaliSense — AI-Powered Handwritten Examination Evaluation**

This notebook demonstrates the HTR pipeline using **TrOCR** (`microsoft/trocr-base-handwritten`):

1. **Line segmentation** — Split a page image into individual text lines
2. **TrOCR inference** — Recognize handwritten text from each line
3. **Confidence scoring** — Model confidence per line
4. **Benchmarking** — CER/WER evaluation on real IAM data

### Model Details
| Property | Value |
|----------|-------|
| Model | `microsoft/trocr-base-handwritten` |
| Architecture | Vision Encoder-Decoder (ViT + GPT-2) |
| Parameters | ~334M |
| Input | RGB image of a single text line |
| Output | Decoded text string + confidence |
| Device | CPU (Intel Core Ultra 9 285H) |

In [ ]:
import sys
sys.path.insert(0, '..')

import time
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Initialize the HTR Pipeline

The pipeline has two components:
- **`HandwritingLineSegmenter`** — Detects and extracts individual text lines from a page
- **`HandwritingRecognizer`** — TrOCR model that reads each line

In [ ]:
from htr import HTRPipeline
from htr.recognizer import HandwritingRecognizer
from htr.line_segmenter import HandwritingLineSegmenter

print('Loading TrOCR model (first time downloads ~330MB)...')
t0 = time.time()

pipeline = HTRPipeline()
# Force model loading
pipeline.recognizer._load_model()

elapsed = time.time() - t0
print(f'Model loaded in {elapsed:.1f}s')
print(f'Device: {pipeline.recognizer.device}')
print(f'Model: {pipeline.recognizer.model_name}')

## 2. Recognize Individual IAM Line Images

Since IAM images are pre-segmented into individual lines, we bypass the line segmenter and feed each image directly to TrOCR.

In [ ]:
# Load IAM metadata
iam_meta = Path('../data/external/iam_lines/metadata.jsonl')
records = []
with open(iam_meta, 'r', encoding='utf-8') as f:
    for line in f:
        rec = json.loads(line.strip())
        if rec.get('split') == 'test' and rec.get('transcription', '').strip():
            records.append(rec)
        if len(records) >= 10:
            break

print(f'Loaded {len(records)} IAM test images for recognition')

In [ ]:
# Run TrOCR on each IAM image
results = []

for i, rec in enumerate(records):
    img_path = Path('../data') / rec['image_file']
    if not img_path.exists():
        print(f'  [{i+1}] Image not found: {img_path}')
        continue
    
    image = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
    ground_truth = rec['transcription']
    
    t0 = time.time()
    recognized = pipeline.recognizer.recognize(image, line_number=i+1)
    elapsed = time.time() - t0
    
    results.append({
        'index': i,
        'ground_truth': ground_truth,
        'predicted': recognized.text,
        'confidence': recognized.confidence,
        'time': elapsed,
        'image_path': str(img_path),
    })
    
    match_symbol = '✓' if ground_truth.lower().strip() == recognized.text.lower().strip() else '≈'
    print(f'  [{i+1:2d}] {match_symbol}  ({elapsed:.2f}s, conf={recognized.confidence:.3f})')
    print(f'       GT:  "{ground_truth}"')
    print(f'       HTR: "{recognized.text}"')

In [ ]:
# Visualize recognition results
n_show = min(6, len(results))
fig, axes = plt.subplots(n_show, 1, figsize=(16, 3 * n_show))
if n_show == 1:
    axes = [axes]

fig.suptitle('TrOCR Recognition Results on IAM Handwriting', fontsize=16, fontweight='bold')

for ax, res in zip(axes, results[:n_show]):
    img = cv2.imread(res['image_path'], cv2.IMREAD_GRAYSCALE)
    ax.imshow(img, cmap='gray', aspect='auto')
    
    gt = res['ground_truth']
    pred = res['predicted']
    conf = res['confidence'] or 0
    color = '#2ECC71' if gt.lower().strip() == pred.lower().strip() else '#E74C3C'
    
    ax.set_title(f'GT: "{gt}"  →  HTR: "{pred}"  (conf={conf:.3f})', 
                 fontsize=10, color=color, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('../data/results/htr_recognition_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. CER and WER Computation

We measure recognition quality using:
- **Character Error Rate (CER):** Levenshtein distance / reference length (character level)
- **Word Error Rate (WER):** Levenshtein distance / reference length (word level)

In [ ]:
def levenshtein(ref, hyp):
    """Compute Levenshtein distance."""
    d = np.zeros((len(ref) + 1, len(hyp) + 1), dtype=int)
    for i in range(len(ref) + 1):
        d[i, 0] = i
    for j in range(len(hyp) + 1):
        d[0, j] = j
    for i in range(1, len(ref) + 1):
        for j in range(1, len(hyp) + 1):
            cost = 0 if ref[i-1] == hyp[j-1] else 1
            d[i, j] = min(d[i-1, j]+1, d[i, j-1]+1, d[i-1, j-1]+cost)
    return d[len(ref), len(hyp)]

def cer(ref, hyp):
    return levenshtein(ref.strip(), hyp.strip()) / max(len(ref.strip()), 1)

def wer(ref, hyp):
    return levenshtein(ref.strip().split(), hyp.strip().split()) / max(len(ref.strip().split()), 1)

# Compute metrics for all results
for r in results:
    r['cer'] = cer(r['ground_truth'], r['predicted'])
    r['wer'] = wer(r['ground_truth'], r['predicted'])

# Summary
avg_cer = np.mean([r['cer'] for r in results])
avg_wer = np.mean([r['wer'] for r in results])
avg_conf = np.mean([r['confidence'] or 0 for r in results])
avg_time = np.mean([r['time'] for r in results])

print('╔══════════════════════════════════════╗')
print('║   HTR Benchmark Results (IAM Data)   ║')
print('╠══════════════════════════════════════╣')
print(f'║  Images tested:    {len(results):>6}           ║')
print(f'║  Average CER:      {avg_cer:>6.4f}  ({avg_cer:.1%})    ║')
print(f'║  Average WER:      {avg_wer:>6.4f}  ({avg_wer:.1%})    ║')
print(f'║  Avg confidence:   {avg_conf:>6.4f}           ║')
print(f'║  Avg time/image:   {avg_time:>5.2f}s            ║')
print('╚══════════════════════════════════════╝')

In [ ]:
# Visualize per-sample metrics
if results:
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
    
    indices = range(len(results))
    cer_vals = [r['cer'] for r in results]
    wer_vals = [r['wer'] for r in results]
    conf_vals = [r['confidence'] or 0 for r in results]
    
    # CER per sample
    colors = ['#2ECC71' if c < 0.1 else '#F39C12' if c < 0.3 else '#E74C3C' for c in cer_vals]
    ax1.bar(indices, cer_vals, color=colors, edgecolor='white')
    ax1.axhline(y=0.1, color='green', linestyle='--', alpha=0.5, label='Good (<10%)')
    ax1.axhline(y=0.3, color='red', linestyle='--', alpha=0.5, label='Poor (>30%)')
    ax1.set_xlabel('Sample Index')
    ax1.set_ylabel('Character Error Rate')
    ax1.set_title('CER per Sample')
    ax1.legend()
    
    # WER per sample
    ax2.bar(indices, wer_vals, color='#4A90D9', edgecolor='white')
    ax2.set_xlabel('Sample Index')
    ax2.set_ylabel('Word Error Rate')
    ax2.set_title('WER per Sample')
    
    # Confidence per sample
    ax3.bar(indices, conf_vals, color='#9B59B6', edgecolor='white')
    ax3.set_xlabel('Sample Index')
    ax3.set_ylabel('Model Confidence')
    ax3.set_title('Recognition Confidence per Sample')
    ax3.set_ylim(0, 1)
    
    plt.suptitle('HTR Performance Metrics', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../data/results/htr_metrics.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. Line Segmentation on Full Pages

For full answer sheets, the pipeline first segments the page into individual lines.

In [ ]:
# Test line segmentation on a generated answer sheet
from htr.line_segmenter import HandwritingLineSegmenter

segmenter = HandwritingLineSegmenter()

test_page = Path('../data/raw/answer_photosynthesis_good.png')
if test_page.exists():
    page_image = cv2.imread(str(test_page))
    
    t0 = time.time()
    regions = segmenter.segment(page_image)
    seg_time = time.time() - t0
    
    print(f'Detected {len(regions)} lines in {seg_time:.2f}s')
    
    # Draw bounding boxes
    overlay = page_image.copy()
    for region in regions:
        x, y, w, h = region.bounding_box
        cv2.rectangle(overlay, (x, y), (x+w, y+h), (0, 0, 255), 2)
        cv2.putText(overlay, f'L{region.line_number}', (x, y-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 10))
    ax1.imshow(cv2.cvtColor(page_image[:1500], cv2.COLOR_BGR2RGB))
    ax1.set_title('Original Page')
    ax1.axis('off')
    
    ax2.imshow(cv2.cvtColor(overlay[:1500], cv2.COLOR_BGR2RGB))
    ax2.set_title(f'Detected Lines ({len(regions)} regions)')
    ax2.axis('off')
    
    plt.suptitle('Line Segmentation', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../data/results/line_segmentation.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Test page image not found. Run: python create_test_images.py')

In [ ]:
# Full pipeline: segmentation + recognition on a page
if test_page.exists():
    print('Running full HTR pipeline on answer sheet...')
    t0 = time.time()
    result = pipeline.recognize_path(test_page)
    total_time = time.time() - t0
    
    print(f'\nPipeline completed in {total_time:.1f}s')
    print(f'Lines detected: {len(result.lines)}')
    print(f'Average confidence: {result.average_confidence:.4f}' if result.average_confidence else 'N/A')
    print(f'\nRecognized text:')
    print('─' * 60)
    for line in result.lines:
        conf_str = f'[{line.confidence:.3f}]' if line.confidence else '[N/A]'
        print(f'  L{line.line_number:2d} {conf_str}: {line.text}')
    print('─' * 60)
    print(f'\nFull text ({len(result.full_text)} chars):')
    print(result.full_text[:500])

## Summary

| Metric | Value |
|--------|-------|
| Model | `microsoft/trocr-base-handwritten` |
| Architecture | Vision Encoder-Decoder (ViT + GPT-2) |
| Device | CPU |
| Inference time | ~1-5s per line |

**Key Observations:**
- TrOCR performs well on clean, single-line handwriting
- Confidence scores correlate with recognition quality
- Line segmentation is critical for full-page processing

**Next:** Proceed to `04_answer_evaluation.ipynb` for rubric-based scoring.